<a href="https://colab.research.google.com/github/hbprosper/dihiggs/blob/main/code/heft_train_bnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Higgs Effective Field Theory (HEFT) Study: Bayesian Neural Network
> Created: Oct 29, 2026 Harrison B. Prosper and Claude<br>

## Introduction

`heft_train.ipynb` finds *one* set of parameters $\omega^*$ of the model `HEFTNet`, by minimising the L1 loss. Here, following Radford Neal [1, 2], we instead treat the 3,944 parameters $\omega$ as unknowns in a Bayesian analysis and *sample* them from their posterior density
$$
p(\omega \mid \mathcal{D}) \propto p(\mathcal{D} \mid \omega)\, \pi(\omega),
$$
given the training data $\mathcal{D}$. The result is a **Bayesian neural network** (BNN): an ensemble of $K$ parameter sets $\omega_1, \ldots, \omega_K$, each saved in its own file `runs/<DIRNAME>/samples/params_NNNN.pth`. The prediction at a point $\boldsymbol{x} = (m_{hh}, \kappa_\lambda, c_t, c_{tt}, c_{ggh}, c_{gghh})$ is the posterior mean
$$
\bar f(\boldsymbol{x}) \approx \frac{1}{K} \sum_{k=1}^{K} f(\boldsymbol{x}; \omega_k),
$$
and the spread of the $f(\boldsymbol{x}; \omega_k)$ measures how well the data determine the function at $\boldsymbol{x}$.

The parameters are sampled with **Hamiltonian Monte Carlo** (HMC), implemented in `dihiggs/hmc.py`. The method, the prior and the likelihood are explained in `docs/bnn/bnn_plan.pdf`; in brief:

* **Prior.** Neal's hierarchical prior: each parameter tensor $g$ (16 of them in `HEFTNet`) has a Gaussian prior of unknown precision $\tau_g$, which is itself sampled.
* **Likelihood.** By default a Laplace density, $p(t \mid f) = e^{-|t - f|/b} / 2b$, with an unknown scale $b$ that is also sampled. Its most probable point is the L1 training of `heft_train.ipynb` with a small weight decay.
* **Start.** From the trained L1 model (*warm start*), so that the chain starts where the posterior is large.
* **Each iteration.** An HMC trajectory of `N_LEAPFROG` steps for $\omega$, followed by exact (Gibbs) draws of the $\tau_g$ and of $b$.

The sampler and the module are checked on problems with known answers in `hmc_toy.ipynb`. Run that first.

**Computing time.** Every HMC step needs the gradient of the posterior over *all* training rows (a random batch would spoil the method), so this notebook is much slower than `heft_train.ipynb` on a CPU. Use a GPU. The notebook times one gradient and prints the expected run time before it starts sampling. A run that is interrupted (`Kernel > Interrupt`, a closed laptop, a time limit) can be continued with `RESUME = True`.

## Bibliography
  1. R. M. Neal, *Bayesian Learning for Neural Networks*, Springer (1996).
  2. R. M. Neal, "MCMC using Hamiltonian dynamics", arXiv:1206.1901 (2011).
  3. M. D. Hoffman and A. Gelman, J. Mach. Learn. Res. 15 (2014) 1593.

In [ ]:
# ------------------------------------------------------------------
# Google Colab setup (nothing happens when run outside Colab)
#   1. clone the dihiggs repository, which holds the data files,
#   2. install the dihiggs package, which also installs drwho,
#   3. make dihiggs/code the working directory so that the relative
#      paths used below (../data, runs/, figures/) work as they do locally.
#
# This notebook needs a GPU: in Colab choose
#   Runtime > Change runtime type > GPU.
# ------------------------------------------------------------------
import os, sys, subprocess, importlib

IN_COLAB = 'google.colab' in sys.modules

# With USE_DRIVE = True the runs/ folder lives in your Google Drive
# (MyDrive/dihiggs/runs). That is where the warm-start model trained by
# heft_train.ipynb is found, and it keeps the samples (and the checkpoint
# from which an interrupted run continues) safe if the runtime disconnects.
USE_DRIVE = True

if IN_COLAB:
    REPO = '/content/dihiggs'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/hbprosper/dihiggs.git', REPO],
                       check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', REPO],
                   check=True)
    importlib.invalidate_caches()
    os.chdir(f'{REPO}/code')

    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_runs = '/content/drive/MyDrive/dihiggs/runs'
        os.makedirs(drive_runs, exist_ok=True)
        if not os.path.exists('runs'):
            os.symlink(drive_runs, 'runs')
        print(f'runs/ is stored in {drive_runs}')

print(f'working directory: {os.getcwd()}')

In [ ]:
import os, sys, time

# the standard module for tabular data
import pandas as pd

# the standard module for array manipulation
import numpy as np

# the standard modules for high-quality plots
import matplotlib as mp
import matplotlib.pyplot as plt

# pytorch
import torch
import torch.nn as nn

# module for shell utilities
import shutil

# ML and other utilities
import drwho.nn as mlp

# Hamiltonian Monte Carlo and the Bayesian neural network
from dihiggs import hmc

# update fonts
plt.rcParams.update({
  "text.usetex": shutil.which('latex') is not None,
  "font.family": "sans-serif",
  "font.size": 12
})

## Computational device

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n\tAvailable device: {str(DEVICE):4s}\n')
if DEVICE.type == 'cuda':
    print(f'\t{torch.cuda.get_device_name(0)}\n')

## Load training data

The same file and the same split as in `heft_train.ipynb`: the first 86.4% of the rows are used for training; the last 12% (the "quick test" rows) are not used here and serve for a first check of the BNN at the end.

In [ ]:
datafile = f'../data/heft_wide_gauss_traindata_13TeV.csv.gz'

print(f'loading {datafile}')
df = pd.read_csv(datafile)

sample_size = len(df)

# Fingerprint of the training file, saved in the configuration below.
import hashlib
with open(datafile, 'rb') as f:
    traindata_sha256 = hashlib.sha256(f.read()).hexdigest()
print(f'sample size: {sample_size}')

# the same split as heft_train.ipynb
train_size = int(0.864*sample_size)
test_size  = int(0.120*sample_size)
val_size   = sample_size - train_size - test_size
print(f'train: {train_size}   val: {val_size}   test: {test_size}')

## Switches

| switch | meaning |
| :--- | :--- |
| `LIKELIHOOD` | `'laplace'` (default; the counterpart of the L1 loss), `'gaussian'` (of the MSE loss), or `'poisson'` (experimental: Gaussian with the *known* POWHEG noise of each bin, the counterpart of the CHI2 loss) |
| `WARM_START` | parameter file of a trained model to start from, or `None` to start from random parameters as Neal does |
| `N_WARMUP` | number of iterations used to tune the step size and the mass matrix; not saved |
| `N_SAMPLES` | number of parameter sets to save |
| `THIN` | save every `THIN`-th iteration after the warm-up |
| `N_LEAPFROG` | number of leapfrog steps per HMC trajectory |
| `TARGET_ACCEPT` | acceptance rate aimed at when tuning the step size |
| `N_CHAINS` | number of independent chains (each in `chain_<c>/`); with more than one, the Gelman–Rubin $\hat R$ is computed |
| `DATA_FRACTION` | fraction of the training *spectra* used (all their bins are kept), for quick tests |
| `SEED` | random seed (chain $c$ uses `SEED + c`) |
| `RESUME` | continue an interrupted run from its checkpoint instead of starting again |

In [ ]:
# ------------------------------------------------------------------
# Switches
# ------------------------------------------------------------------
LIKELIHOOD    = 'laplace'   # 'laplace', 'gaussian' or 'poisson'
WARM_START    = 'runs/L1Loss/dihiggs_params.pth'   # or None
N_WARMUP      = 500
N_SAMPLES     = 1000
THIN          = 1
N_LEAPFROG    = 100
TARGET_ACCEPT = 0.8
N_CHAINS      = 1
DATA_FRACTION = 1.0
SEED          = 42
RESUME        = False

# rows per chunk when the gradient is summed over all training rows;
# lower it if the GPU runs out of memory
CHUNK_SIZE    = 2**17

assert LIKELIHOOD in ('laplace', 'gaussian', 'poisson'), \
    f'unknown likelihood {LIKELIHOOD}'

# the folder for this run: runs/<DIRNAME>
DIRNAME = f'BNN_{LIKELIHOOD}'
NAME    = 'dihiggs'
BASE    = f'runs/{DIRNAME}'
print(f'run folder: {BASE}')

## Estimated noise in each bin

Only the `'poisson'` likelihood uses this, and it uses it as a *known* property of each training row, not as a target. It is the estimate of `heft_train.ipynb` (where it is explained): the POWHEG spectra behave as histograms of $N_\text{eff}$ equal-weight events, so that the variance of a bin with cross section $f_k$ in a spectrum of total cross section $\sigma_\text{tot}$ is $\sigma_k^2 = (f_k + \sigma_\text{tot}/N_\text{eff})\, \sigma_\text{tot}/N_\text{eff}$, including the variance of one extra event.

In [ ]:
# total cross section of the spectrum to which each row belongs
df['sigma_tot'] = df.groupby('ID')['sigma'].transform('sum')

spectra_train = df.pivot_table(index='ID', columns='mhh', values='sigma',
                               fill_value=0.0)
F     = spectra_train.to_numpy(np.float64)
F_tot = F.sum(axis=1, keepdims=True)

# bin-to-bin jitter: each bin minus the average of its two neighbours
f_mid, f_lo, f_hi = F[:, 1:-1], F[:, :-2], F[:, 2:]
D = f_mid - (f_lo + f_hi) / 2
good = (f_mid > 0) & (f_lo > 0) & (f_hi > 0)
q = np.full_like(D, np.nan)
q[good] = D[good]**2 / (F_tot * (f_mid + (f_lo + f_hi) / 4))[good]
smooth = spectra_train.columns.to_numpy()[1:-1] >= (38 - 17 + 0.5) / 100
N_EFF  = 0.4549 / np.nanmedian(q[:, smooth])

one_event = df.sigma_tot / N_EFF
df['noise_var'] = (df.sigma + one_event) * one_event
print(f'effective number of events per spectrum: N_eff = {N_EFF:.0f}')

## The training data as tensors

The inputs `X` are the six features; the target `Y` is the POWHEG cross section alone, as in `heft_train.ipynb`. (The noise variance of a bin is not a target: for the scaled Poisson counts of POWHEG it is fixed by the cross section itself, $\text{Var}(\sigma_k) = w_j \sigma_k$, so there is nothing to learn. With `'laplace'` and `'gaussian'` the noise is described by the likelihood's scale $b$, which is sampled; with `'poisson'` the estimated variances above are handed to the likelihood directly, as `v_train`.) With `DATA_FRACTION < 1`, a random fraction of the spectra in the training rows is kept, all their bins included.

In [ ]:
from dihiggs import heftnet as NN

X = df[NN.FEATURES].to_numpy(dtype=np.float32)
# Target: the POWHEG cross section per bin (pb), one column, shape [N, 1]
Y = df[[NN.TARGET]].to_numpy(dtype=np.float32)
ID = df['ID'].to_numpy()

# training rows: the first train_size rows, as in heft_train.ipynb
rows = np.arange(train_size)
if DATA_FRACTION < 1:
    ids  = np.unique(ID[rows])
    keep = np.random.RandomState(SEED).choice(
        ids, size=max(1, int(DATA_FRACTION * len(ids))), replace=False)
    rows = rows[np.isin(ID[rows], keep)]
    print(f'using {len(keep)} of {len(ids)} spectra')

# torch.from_numpy shares memory with the numpy array; .to(DEVICE) copies
# the tensor to the GPU (if there is one)
x_train = torch.from_numpy(X[rows]).to(DEVICE)
t_train = torch.from_numpy(Y[rows]).to(DEVICE)

# the estimated noise variance of each training row, in the same order;
# used only by the 'poisson' likelihood
v_train = torch.from_numpy(df['noise_var'].to_numpy(np.float32)[rows]).to(DEVICE)

# the quick test rows (not used for sampling)
test = slice(train_size + val_size, train_size + val_size + test_size)
x_test = torch.from_numpy(X[test]).to(DEVICE)
t_test = Y[test, 0]

print(f"""
    Features: {NN.FEATURES}
    training rows: {len(x_train)}
    test rows:     {len(x_test)}
""")

## Model and warm start

The model is `HEFTNet` from `heftnet.py`, exactly as in `heft_train.ipynb`. With a warm start, its parameters are those of a trained L1 model. The file is only read, never written.

In [ ]:
def new_model(chain=0):
    # A HEFTNet with either the warm-start parameters or, if WARM_START is
    # None, random initial parameters (different for each chain).
    torch.manual_seed(SEED + chain)
    model = NN.HEFTNet().to(DEVICE)
    if WARM_START is not None:
        # HEFTNet.load reads a state_dict saved with torch.save
        model.load(WARM_START)
        model.to(DEVICE)
    return model

model = new_model()
print(model)
print(f'Number of parameters: {mlp.number_of_parameters(model)}')

## Configuration

`mlp.Config` records the settings of this run in `runs/<DIRNAME>/dihiggs_config.yaml`.

In [ ]:
if RESUME:
    config = mlp.Config(f'{BASE}/{NAME}_config.yaml')
else:
    config = mlp.Config(NAME, dirname=DIRNAME)
    config('model_name', NN.NAME)
    config('features', NN.FEATURES)
    config('target', NN.TARGET)
    config('traindata_sha256', traindata_sha256)
    config('train_size', train_size)
    config('n_train_rows', len(x_train))

    config('bnn/likelihood', LIKELIHOOD)
    config('bnn/warm_start', str(WARM_START))
    config('bnn/n_warmup', N_WARMUP)
    config('bnn/n_samples', N_SAMPLES)
    config('bnn/thin', THIN)
    config('bnn/n_leapfrog', N_LEAPFROG)
    config('bnn/target_accept', TARGET_ACCEPT)
    config('bnn/n_chains', N_CHAINS)
    config('bnn/data_fraction', DATA_FRACTION)
    if LIKELIHOOD == 'poisson':
        config('bnn/N_eff', float(N_EFF))
    config('bnn/seed', SEED)
    config.save()
print(config)

## The parameters, found automatically

`hmc.ParameterView` asks the model for its parameters (`named_parameters()`) and makes each tensor a group of Neal's prior. The names say where each tensor lives: `P.0.weight` is the weight matrix of item 0 of the `nn.Sequential` called `P`.

In [ ]:
view = hmc.ParameterView(model)
print(view.summary())

## Prior, likelihood and posterior

`hmc.Posterior` computes the potential energy $U(\omega) = -\log p(\mathcal{D} \mid \omega) - \log \pi(\omega)$ and its gradient, summed over all training rows in chunks of `CHUNK_SIZE`. One gradient is timed to predict the run time.

In [ ]:
def make_posterior(model):
    view  = hmc.ParameterView(model)
    prior = hmc.HierarchicalPrior(view)
    if LIKELIHOOD == 'laplace':
        lik = hmc.LaplaceLikelihood()
    elif LIKELIHOOD == 'gaussian':
        lik = hmc.GaussianLikelihood()
    else:
        # the known variances are given to the likelihood, not as a target
        lik = hmc.PoissonLikelihood(v_train)
    return hmc.Posterior(model, lik, prior, x_train, t_train,
                         chunk_size=CHUNK_SIZE)

post = make_posterior(model)

seconds = post.time_gradient(post.view.get())
n_grad  = (N_WARMUP + N_SAMPLES * THIN) * (N_LEAPFROG + 1) * N_CHAINS
print(f'one gradient over {len(x_train)} rows: {seconds*1000:.1f} ms')
print(f'expected run time: about {n_grad * seconds / 3600:.1f} hours '
      f'({n_grad} gradients)')

## Sample

Each chain writes into its own folder (`runs/<DIRNAME>` for a single chain, `runs/<DIRNAME>/chain_<c>` otherwise):

| file | contents |
| :--- | :--- |
| `samples/params_0001.pth`, ... | one sampled parameter set each, readable with `HEFTNet().load(file)` |
| `samples/index.csv` | for each sample: file, iteration, $U$, noise scale $b$, precisions $\tau_g$ |
| `dihiggs_hmc_trace.csv` | one row per iteration: accepted?, $\Delta H$, $U$, step size, $b$, $\tau_g$, and the monitor values |
| `dihiggs_hmc_state.pth` | the checkpoint from which `RESUME = True` continues |

The **monitor values** `m_0` and `m_1` are the predictions, for the Standard Model point $(\kappa_\lambda, c_t, c_{tt}, c_{ggh}, c_{gghh}) = (1, 1, 0, 0, 0)$, in the first bin above threshold (262.5 GeV) and in the bin at 997.5 GeV. Their traces show whether the chain mixes for quantities we care about. (Individual parameters need not converge, because permuting hidden units leaves the network unchanged.)

In [ ]:
# the SM point in two bins: x = (k - 17 + 0.5)/100 for POWHEG bin k
x_monitor = torch.tensor([[0.005, 1.0, 1.0, 0.0, 0.0, 0.0],
                          [0.495, 1.0, 1.0, 0.0, 0.0, 0.0]],
                         device=DEVICE)
monitor = lambda model: model(x_monitor)

def chain_dir(c):
    return BASE if N_CHAINS == 1 else f'{BASE}/chain_{c+1}'

samplers = []
for c in range(N_CHAINS):
    print(f'\n--- chain {c+1} of {N_CHAINS} ---')
    model_c = new_model(c)
    post_c  = make_posterior(model_c)
    sampler = hmc.HMC(post_c, chain_dir(c),
                      n_leapfrog=N_LEAPFROG,
                      target_accept=TARGET_ACCEPT,
                      seed=SEED + c,
                      name=NAME,
                      monitor=monitor)
    sampler.run(n_warmup=N_WARMUP, n_samples=N_SAMPLES, thin=THIN,
                print_every=10, resume=RESUME)
    samplers.append(sampler)

## Diagnostics

* **Traces.** After the warm-up (dashed line), $U$, the noise scale $b$ and the monitor values should fluctuate about a constant level, with no drift.
* **Acceptance rate** between about 0.6 and 0.9, and few or no divergences.
* **Effective sample size** $n_\text{eff}$: the number of independent draws worth of information in the correlated chain. We aim for at least 100 for the monitor values.
* **$\hat R$** (several chains only): close to 1, say below 1.05.

In [ ]:
traces = [s.trace() for s in samplers]

fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
axes = axes.ravel()
for c, tr in enumerate(traces):
    n_warm = np.sum(tr['phase'] == 'warmup')
    for ax, key, label in zip(axes, ['U', 'scale', 'm_0', 'm_1'],
                              [r'$U$', r'noise scale $b$ (pb)',
                               r'SM, 262.5 GeV (pb)', r'SM, 997.5 GeV (pb)']):
        ax.plot(tr['iteration'], tr[key], lw=0.6, label=f'chain {c+1}')
        ax.axvline(n_warm, color='gray', ls='--')
        ax.set_xlabel('iteration')
        ax.set_ylabel(label)
        ax.ticklabel_format(axis='y', scilimits=[-2, 3])
if N_CHAINS > 1:
    axes[0].legend(fontsize=9)
fig.tight_layout()
plt.savefig(f'{BASE}/{NAME}_hmc_traces.png')
plt.show()

for c, (s, tr) in enumerate(zip(samplers, traces)):
    after = tr['phase'] == 'sample'
    print(f'chain {c+1}: acceptance rate {s.acceptance_rate():.3f}, '
          f'step size {s.eps:.3e}, divergences {s.n_diverge}')
    for key in ['U', 'scale', 'm_0', 'm_1']:
        print(f'    n_eff({key:5s}) = {hmc.ess(tr[key][after]):7.0f}'
              f'  of {after.sum()}')

if N_CHAINS > 1:
    n = min(np.sum(tr['phase'] == 'sample') for tr in traces)
    for key in ['U', 'scale', 'm_0', 'm_1']:
        chains = [tr[key][tr['phase'] == 'sample'][:n] for tr in traces]
        print(f'R-hat({key:5s}) = {hmc.rhat(chains):.3f}')

## The Bayesian neural network

`hmc.BNN` reads every `samples/params_*.pth` in the run folder (from all chains) and computes, at any set of points, the posterior mean, standard deviation, quantiles and credible intervals of the output.

### Quick test: BNN mean against POWHEG

On the quick test rows, compare the posterior mean with POWHEG, as in `heft_train.ipynb`, and with the warm-start model.

In [ ]:
bnn = hmc.BNN(NN.HEFTNet, BASE, device=DEVICE)
print(f'number of sampled parameter sets: {len(bnn)}')

y_bnn = bnn.mean(x_test).cpu().numpy()
s_bnn = bnn.std(x_test).cpu().numpy()

with torch.no_grad():
    y_l1 = new_model()(x_test).cpu().numpy()

print(f"""
    mean |BNN mean - POWHEG|:   {np.mean(np.abs(y_bnn - t_test)):.4e} pb
    mean |warm start - POWHEG|: {np.mean(np.abs(y_l1 - t_test)):.4e} pb
    median posterior std/|BNN mean|: {np.median(s_bnn / np.abs(y_bnn)):.2e}
""")

def plot_results(y, t, filename, xmin=1e-6, xmax=1e2, ymin=1e-6, ymax=1e2,
                 ftsize=14):
    fig = plt.figure(figsize=(8.4, 4))
    for i in range(2):
        ax = fig.add_subplot(1, 2, i+1)
        if i < 1:
            ax.set_xlim(0, 25)
            ax.set_ylim(0, 25)
        else:
            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)
            ax.set_xscale('log')
            ax.set_yscale('log')
        ax.set_xlabel(r'$\sigma$ (POWHEG)', fontsize=ftsize)
        ax.set_ylabel(r'$\sigma$ (BNN mean)', fontsize=ftsize)
        ax.plot(t, y, 'b', marker='.', markersize=2, linestyle='')
        ax.plot([xmin, xmax], [ymin, ymax], linestyle='solid', color='red')
        ax.grid(True, which="both", linestyle='-')
    fig.tight_layout()
    plt.savefig(filename)
    plt.show()

plot_results(y_bnn, t_test, f'{BASE}/{NAME}_plots.png')

### Spectra with credible bands

A few holdout spectra (never used in training), with the BNN posterior mean and its central 68% and 95% credible bands. The lower panels show the band relative to the mean, which makes it visible where the data determine the function least well.

In [ ]:
spectra = pd.read_csv('../data/heft_wide_gauss_testdata_13TeV.csv')
params  = ['klambda', 'CT', 'CTT', 'CGGH', 'CGGHH']
bins    = [c for c in spectra.columns if c.isdigit()]
x_bins  = (np.arange(len(bins)) + 0.5) / 100          # bin centres in x
mhh     = 255 + 1500 * x_bins                         # in GeV

def spectrum_inputs(row):
    theta = spectra[params].iloc[row].to_numpy(dtype=np.float32)
    inputs = np.column_stack([x_bins.astype(np.float32),
                              np.tile(theta, (len(x_bins), 1))])
    return torch.from_numpy(inputs).to(DEVICE)

rows_to_plot = [0, 1, 2, 3, 4, 5, 6, 7]
fig, axes = plt.subplots(4, 4, figsize=(13, 9),
                         gridspec_kw={'height_ratios': [3, 1, 3, 1]})
for n, row in enumerate(rows_to_plot):
    ax  = axes[2 * (n // 4), n % 4]
    axr = axes[2 * (n // 4) + 1, n % 4]
    inputs = spectrum_inputs(row)
    q = bnn.quantiles(inputs, [0.025, 0.16, 0.5, 0.84, 0.975]).cpu().numpy()
    mean = bnn.mean(inputs).cpu().numpy()
    powheg = spectra[bins].iloc[row].to_numpy()

    ax.fill_between(mhh, q[0], q[4], color='steelblue', alpha=0.25, lw=0)
    ax.fill_between(mhh, q[1], q[3], color='steelblue', alpha=0.5, lw=0)
    ax.plot(mhh, mean, 'b', lw=1)
    ax.step(mhh, powheg, 'k', where='mid', lw=0.7)
    ax.set_ylim(0, 1.3 * max(powheg.max(), q[4].max()))
    ax.ticklabel_format(axis='y', scilimits=[-2, 2])

    # relative band, (quantile - mean)/mean, shown only where the mean is
    # above 1% of its maximum (near zero the ratio is meaningless)
    scale = np.where(mean > 0.01 * mean.max(), mean, np.nan)
    axr.fill_between(mhh, (q[0] - mean) / scale, (q[4] - mean) / scale,
                     color='steelblue', alpha=0.25, lw=0)
    axr.fill_between(mhh, (q[1] - mean) / scale, (q[3] - mean) / scale,
                     color='steelblue', alpha=0.5, lw=0)
    axr.axhline(0, color='b', lw=0.8)
    axr.ticklabel_format(axis='y', scilimits=[-2, 2])
    if n % 4 == 0:
        ax.set_ylabel(r'$\sigma$ (pb)')
        axr.set_ylabel('rel.')
    axr.set_xlabel(r'$m_{hh}$ (GeV)')
fig.tight_layout()
plt.savefig(f'{BASE}/{NAME}_spectra_bands.png')
plt.show()

### The coefficient functions $a_i(m_{hh})$ with credible bands

`bnn.apply` evaluates any function of the model for every sample; here `model.coeffs`, which returns the 23 functions $a_i(m_{hh})$.

In [ ]:
xs = torch.linspace(0.0, 0.8, 161, device=DEVICE)
A  = bnn.apply(lambda m, x: m.coeffs(x), xs).cpu().numpy()   # [K, 161, 23]
lo, med, hi = np.quantile(A, [0.025, 0.5, 0.975], axis=0)
m_axis = 255 + 1500 * xs.cpu().numpy()

fig, axes = plt.subplots(6, 4, figsize=(12, 13), sharex=True)
axes = axes.ravel()
for i in range(23):
    ax = axes[i]
    ax.fill_between(m_axis, lo[:, i], hi[:, i], color='steelblue', alpha=0.4,
                    lw=0)
    ax.plot(m_axis, med[:, i], 'b', lw=1)
    ax.set_title(f'$a_{{{i+1}}}$', fontsize=11)
    ax.ticklabel_format(axis='y', scilimits=[-2, 2])
axes[23].axis('off')
for ax in axes[19:23]:
    ax.set_xlabel(r'$m_{hh}$ (GeV)')
fig.tight_layout()
plt.savefig(f'{BASE}/{NAME}_coefficients_bands.png')
plt.show()